# Compound Comparators

## The Problem: Fields That Belong Together

Per-field comparators score each field independently. That works for most fields,
but sometimes fields only make sense as a group.

**Example:** A person has `surname` and `name`. Gold is "Smith, John", extracted is
"Smith, Jane". Independent scoring gives:
- `surname`: "Smith" vs "Smith" -> match (score 1.0)
- `name`: "John" vs "Jane" -> mismatch (score 0.0)

The extractor gets credit for "Smith" -- but "Smith, Jane" is a **completely different
person** from "Smith, John". The surname match is misleading.

## The Solution: Compound Comparators

A compound comparator is a special kind of **batch comparator** (introduced in
`04_example_semantic`). Both receive multiple fields at once instead of scoring
one field at a time, but they serve different purposes:

| | General batch comparator (e.g. `semantic`) | Compound comparator                                                                                                   |
|---|---|-----------------------------------------------------------------------------------------------------------------------|
| **Groups by** | Comparator name -- all fields using `"semantic"` in one record are batched together | **Parent path** -- fields are grouped by their parent object (e.g. `student.surname` + `student.name`)                |
| **Scores** | Each field independently (one score per field) | The **group as a unit** -- one joint decision for all fields in the group                                             |
| **Typical use** | Send multiple fields to an LLM in one API call for efficiency | Fields that are semantically coupled (name parts, value + unit, address components)                                   |
| **Field count in metrics** | All fields count | Only the **primary** field counts; others become `skipped`, so that the compound fields only get one score as a whole |
| **Base class** | Implement the batch protocol directly | Subclass `CompoundComparator` -- handles grouping, primary/skip, incomplete groups                                    |

Key concepts:
- **Fields** -- which sibling field names form the compound (e.g. `["surname", "name"]`)
- **Primary** -- one field gets the score; the others become `skipped` (so they don't
  double-count in metrics)
- **Group by parent** -- when the same fields appear in multiple places (e.g. student
  vs teacher), each parent gets its own group

## Data

Each record has a student and a teacher, each with `surname`, `name`, and `gender`.
The compound comparator only groups `surname` + `name` -- `gender` is scored
independently with `exact`.

In [1]:
GOLD = [
    {
        "student": {"surname": "Smith", "name": "John", "gender": "male"},
        "teacher": {"surname": "Chen", "name": "Wei", "gender": "female"},
    },
    {
        "student": {"surname": "Kim", "name": "Soo", "gender": "female"},
        "teacher": {"surname": "Mueller", "name": "Anna", "gender": "female"},
    },
]

EXTRACTED = [
    {   # student name wrong (and gender changed to match the wrong name)
        "student": {"surname": "Smith", "name": "Jane", "gender": "female"},
        "teacher": {"surname": "Chen", "name": "Wei", "gender": "female"},
    },
    {   # student correct, teacher surname wrong
        "student": {"surname": "Kim", "name": "Soo", "gender": "female"},
        "teacher": {"surname": "Schmidt", "name": "Anna", "gender": "female"},
    },
]

## Method 1: User `exact` Comparator: Independent Scoring (the problem)

Each field scored on its own with `exact`.

In [2]:
from struct_extract_eval import evaluate
from example_utils import show_run

PERSON_FIELDS = {
    "surname": {"type": "string", "x-eval-compare": "exact"},
    "name": {"type": "string", "x-eval-compare": "exact"},
    "gender": {"type": "string", "x-eval-compare": "exact"},
}
independent_schema = {
    "type": "object",
    "properties": {
        "student": {"type": "object", "properties": dict(PERSON_FIELDS)},
        "teacher": {"type": "object", "properties": dict(PERSON_FIELDS)},
    },
}

run_independent = evaluate(GOLD, EXTRACTED, schema=independent_schema)

show_run(run_independent, "Independent scoring")


Independent scoring
  mean P=0.75  R=0.75  F1=0.75   (2 record(s))
  record  path             gold       extracted  score  status    reason
  0       student.surname  'Smith'    'Smith'    1.0    match
  0       student.name     'John'     'Jane'     0.0    mismatch  mismatch
  0       student.gender   'male'     'female'   0.0    mismatch  mismatch
  0       teacher.surname  'Chen'     'Chen'     1.0    match
  0       teacher.name     'Wei'      'Wei'      1.0    match
  0       teacher.gender   'female'   'female'   1.0    match
  1       student.surname  'Kim'      'Kim'      1.0    match
  1       student.name     'Soo'      'Soo'      1.0    match
  1       student.gender   'female'   'female'   1.0    match
  1       teacher.surname  'Mueller'  'Schmidt'  0.0    mismatch  mismatch
  1       teacher.name     'Anna'     'Anna'     1.0    match
  1       teacher.gender   'female'   'female'   1.0    match


Record 0: `student.surname`="Smith" matches, giving misleading credit.
But "Smith, Jane" is not "Smith, John" -- the full name is wrong.

## Method 2: Compound Comparator

step 1 : Define a Compound Comparator

Subclass `CompoundComparator` and implement `compare()`. The base class handles
grouping by parent, primary/skip logic, and incomplete groups.

- `fields`: which sibling field names form the compound
- `primary`: which field gets the score (others become `skipped`)
- `compare(gold, extracted)`: receives dicts of `{field_name: value}`, returns a score

In [3]:
from struct_extract_eval import CompoundComparator


class NameCompoundComparator(CompoundComparator):
    """Score surname + name as one full name."""

    def __init__(self) -> None:
        super().__init__(
            fields=["surname", "name"],
            primary="surname",        # surname gets the score, name becomes skipped
            name="name_compound",
        )

    def compare(self, gold: dict[str, object], extracted: dict[str, object]) -> float:
        """Compare two full names. Case-insensitive."""
        gold_full = f"{gold['name']} {gold['surname']}"
        ext_full = f"{extracted['name']} {extracted['surname']}"
        return 1.0 if gold_full == ext_full else 0.0

Step 2: Register and Evaluate

Register the comparator, then set `surname` and `name` to use it. `gender` stays
as `exact` -- it's not part of the compound group.

In [4]:
from struct_extract_eval.core.comparators.registry import register

register("name_compound", NameCompoundComparator(), overwrite=True)

# surname + name use compound; gender is scored independently
COMPOUND_PERSON_FIELDS = {
    "surname": {"type": "string", "x-eval-compare": "name_compound"},
    "name": {"type": "string", "x-eval-compare": "name_compound"},
    "gender": {"type": "string", "x-eval-compare": "exact"},
}
compound_schema = {
    "type": "object",
    "properties": {
        "student": {"type": "object", "properties": dict(COMPOUND_PERSON_FIELDS)},
        "teacher": {"type": "object", "properties": dict(COMPOUND_PERSON_FIELDS)},
    },
}

run_compound = evaluate(GOLD, EXTRACTED, schema=compound_schema)

show_run(run_compound, "Compound scoring")


Compound scoring
  mean P=0.62  R=0.62  F1=0.62   (2 record(s))
  record  path             gold       extracted  score  status    reason
  0       student.surname  'Smith'    'Smith'    0.0    mismatch  compound: mismatch
  0       student.name     'John'     'Jane'     0.0    skipped   compound with student.surname
  0       student.gender   'male'     'female'   0.0    mismatch  mismatch
  0       teacher.surname  'Chen'     'Chen'     1.0    match     compound: match
  0       teacher.name     'Wei'      'Wei'      0.0    skipped   compound with teacher.surname
  0       teacher.gender   'female'   'female'   1.0    match
  1       student.surname  'Kim'      'Kim'      1.0    match     compound: match
  1       student.name     'Soo'      'Soo'      0.0    skipped   compound with student.surname
  1       student.gender   'female'   'female'   1.0    match
  1       teacher.surname  'Mueller'  'Schmidt'  0.0    mismatch  compound: mismatch
  1       teacher.name     'Anna'     'Ann

### What Changed

- `student.surname` now scores the **full name** ("John Smith" vs "Jane Smith" -> mismatch).
  No more misleading credit for a matching surname.
- `student.name` is `skipped` -- it's the supporting field. It contributed to the compound
  decision but doesn't count separately in metrics.
- `student.gender` is scored independently with `exact` -- it's not part of the compound.
  Only `surname` and `name` are grouped together.
- Total scored fields: 4 per record (surname + gender per person). Without compound it
  would be 6 (surname + name + gender per person).

### How Grouping Works

The comparator receives the `name_compound` fields per record and groups them by
**parent path**. Fields using other comparators (like `gender` with `exact`) are
not affected:

```
student.surname + student.name  ->  parent "student"  ->  one full name comparison
teacher.surname + teacher.name  ->  parent "teacher"  ->  another full name comparison
student.gender                  ->  scored independently with exact
teacher.gender                  ->  scored independently with exact
```

## Compound Inside Arrays

When the same fields appear inside array elements, instance paths (`students[0]`,
`students[1]`) ensure each element gets its own group.

In [5]:
GOLD_ARRAY = [
    {"students": [
        {"surname": "Smith", "name": "John", "gender": "male"},
        {"surname": "Kim", "name": "Soo", "gender": "female"},
    ]},
]

EXTRACTED_ARRAY = [
    {"students": [
        {"surname": "Smith", "name": "Jane", "gender": "female"},  # wrong name
        {"surname": "Kim", "name": "Soo", "gender": "female"},     # correct
    ]},
]

array_schema = {
    "type": "object",
    "properties": {
        "students": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "surname": {"type": "string", "x-eval-compare": "name_compound"},
                    "name": {"type": "string", "x-eval-compare": "name_compound"},
                    "gender": {"type": "string", "x-eval-compare": "exact"},
                },
            },
        },
    },
}

run_array = evaluate(GOLD_ARRAY, EXTRACTED_ARRAY, schema=array_schema)

show_run(run_array, "Compound inside array")


Compound inside array
  mean P=0.50  R=0.50  F1=0.50   (1 record(s))
  record  path                 gold      extracted  score  status    reason
  0       students[0].surname  'Smith'   'Smith'    0.0    mismatch  compound: mismatch
  0       students[0].name     'John'    'Jane'     0.0    skipped   compound with students[0].surname
  0       students[0].gender   'male'    'female'   0.0    mismatch  mismatch
  0       students[1].surname  'Kim'     'Kim'      1.0    match     compound: match
  0       students[1].name     'Soo'     'Soo'      0.0    skipped   compound with students[1].surname
  0       students[1].gender   'female'  'female'   1.0    match
